<a href="https://colab.research.google.com/github/ZahinAbrar/ASU-Dissertation-Template/blob/main/A_Simple_GNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
!pip install tensorflow spektral
!pip install networkx scikit-learn


In [13]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Download and prepare Cora dataset manually
def load_cora_dataset():
    # URL for Cora dataset
    cora_content_url = 'https://raw.githubusercontent.com/kimiyoung/planetoid/master/data/cora/cora.content'
    cora_cite_url = 'https://raw.githubusercontent.com/kimiyoung/planetoid/master/data/cora/cora.cites'

    # Download content
    import urllib.request
    content_path = '/tmp/cora.content'
    cite_path = '/tmp/cora.cites'

    urllib.request.urlretrieve(cora_content_url, content_path)
    urllib.request.urlretrieve(cora_cite_url, cite_path)

    # Read content file
    node_features = {}
    node_labels = {}

    with open(content_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            node_id = parts[0]
            features = list(map(float, parts[1:-1]))
            label = parts[-1]
            node_features[node_id] = features
            node_labels[node_id] = label

    # Create graph
    G = nx.read_edgelist(cite_path)

    # Prepare features matrix
    nodes = list(G.nodes())
    features = np.array([node_features[node] for node in nodes])

    # Prepare labels
    unique_labels = sorted(set(node_labels.values()))
    label_to_index = {label: i for i, label in enumerate(unique_labels)}
    labels = np.array([label_to_index[node_labels[node]] for node in nodes])

    # Create adjacency matrix
    adj_matrix = nx.adjacency_matrix(G, nodelist=nodes).toarray()

    return features, labels, adj_matrix, unique_labels

# Load and prepare the dataset
features, labels, adj_matrix, label_names = load_cora_dataset()

# Preprocess features
scaler = StandardScaler()
features = scaler.fit_transform(features)

# Prepare train/val/test splits
num_nodes = features.shape[0]
train_mask = np.zeros(num_nodes, dtype=bool)
val_mask = np.zeros(num_nodes, dtype=bool)
test_mask = np.zeros(num_nodes, dtype=bool)

# Custom train/val/test split
np.random.seed(42)
indices = np.random.permutation(num_nodes)
train_idx = indices[:140]
val_idx = indices[140:640]
test_idx = indices[640:]

train_mask[train_idx] = True
val_mask[val_idx] = True
test_mask[test_idx] = True

# Convert to TensorFlow tensors
features = tf.convert_to_tensor(features, dtype=tf.float32)
labels = tf.convert_to_tensor(labels, dtype=tf.int32)
adj_matrix = tf.convert_to_tensor(adj_matrix, dtype=tf.float32)

# Number of features and classes
num_features = features.shape[1]
num_classes = len(label_names)

# Define the Graph Neural Network model
class GNNModel(keras.Model):
    def __init__(self, n_hidden, n_classes):
        super(GNNModel, self).__init__()

        # First graph convolution layer
        self.conv1 = keras.layers.Dense(n_hidden, activation='relu')

        # Dropout layer
        self.dropout = keras.layers.Dropout(0.5)

        # Output layer
        self.conv2 = keras.layers.Dense(n_classes, activation='softmax')

    def call(self, inputs, training=False):
        x, adj = inputs

        # Perform graph convolution manually (message passing)
        x = tf.matmul(adj, x)

        # First layer
        x = self.conv1(x)
        x = self.dropout(x, training=training)

        # Output layer
        x = self.conv2(x)

        return x

# Hyperparameters
learning_rate = 0.01
epochs = 200

# Model and optimizer
model = GNNModel(16, num_classes)
optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
loss_fn = keras.losses.SparseCategoricalCrossentropy()

# Training function
@tf.function
def train_step(features, adj_matrix, labels, mask):
    with tf.GradientTape() as tape:
        # Make predictions
        predictions = model([features, adj_matrix], training=True)

        # Mask the labels and predictions
        masked_labels = tf.boolean_mask(labels, mask)
        masked_predictions = tf.boolean_mask(predictions, mask)

        # Compute loss
        loss = loss_fn(masked_labels, masked_predictions)

    # Compute gradients
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))

    return loss

# Evaluation function
def evaluate(features, adj_matrix, labels, mask):
    predictions = model([features, adj_matrix], training=False)

    masked_labels = tf.boolean_mask(labels, mask)
    masked_predictions = tf.boolean_mask(predictions, mask)

    accuracy = keras.metrics.sparse_categorical_accuracy(
        masked_labels, masked_predictions
    )
    return tf.reduce_mean(accuracy)

# Training loop
print("Starting training...")
for epoch in range(epochs):
    # Training step
    loss = train_step(features, adj_matrix, labels, train_mask)

    # Validation accuracy
    val_accuracy = evaluate(features, adj_matrix, labels, val_mask)

    # Print progress every 10 epochs
    if epoch % 10 == 0:
        print(f'Epoch: {epoch}, Loss: {loss.numpy():.4f}, '
              f'Validation Accuracy: {val_accuracy.numpy():.4f}')

# Final evaluation
test_accuracy = evaluate(features, adj_matrix, labels, test_mask)
print(f'\nFinal Test Accuracy: {test_accuracy.numpy():.4f}')
print(f'Number of classes: {num_classes}')
print(f'Class names: {label_names}')



HTTPError: HTTP Error 404: Not Found